# ニューラルネットワーク入門 — NumPy でゼロから作る

このノートブックでは、深層学習ライブラリを一切使わず、**NumPy の行列計算だけ**でニューラルネットワークを作ります。
最後には、自分で書いたネットワークで**手書き数字の分類**(正解率 95% 以上)まで到達します。

学ぶ内容:

1. ニューロンと活性化関数
2. 順伝播(行列のかけ算で予測する)
3. 損失関数(どれくらい間違えたかを測る)
4. 勾配降下法(損失が減る方向に進む)
5. 逆伝播(全パラメータの勾配を一度に求める)
6. 手書き数字の分類

## このノートブックの使い方

- コードセルをクリックして **Shift + Enter** を押すと、そのセルが実行され、次のセルに移動します。
- **上から順番に** 実行してください。前のセルで作った関数を、後のセルで使います。
- **最初のセルは日本語フォントの読み込みのため、実行に時間がかかります**(数十秒程度)。`[*]` の表示が数字に変わるまで待ってください。

In [ ]:
import piplite
await piplite.install("matplotlib-fontja==1.1.0")

import matplotlib_fontja
import matplotlib.pyplot as plt
import numpy as np

matplotlib_fontja.japanize()
np.set_printoptions(precision=3, suppress=True)

## 1. ニューロン — いちばん小さな部品

ニューラルネットワークは「ニューロン」という小さな計算単位の集まりです。
1 つのニューロンがやることは、とても単純です。

1. 各入力 $x_i$ に**重み** $w_i$ を掛けて全部足す
2. **バイアス** $b$ を足す(この結果を $z$ と呼びます)
3. **活性化関数**に通して出力にする

$$z = w_1 x_1 + w_2 x_2 + \cdots + b$$

重みは「その入力をどれくらい重視するか」、バイアスは「どれくらい反応しやすいか」を表す数で、
**学習とはこの重みとバイアスを少しずつ調整すること**です。まず 1 に相当する計算をやってみます。

In [ ]:
x = np.array([1.0, 0.5])   # 入力(特徴量が 2 つ)
w = np.array([0.8, -0.4])  # 重み
b = 0.1                    # バイアス

z = np.dot(w, x) + b       # 重み付き和
print("重み付き和 z =", z)

## 2. 活性化関数 — 「非線形」が賢さのもと

重み付き和 $z$ をそのまま出力すると、何層重ねてもただの 1 次式(直線)にしかなりません。
そこで $z$ を**曲がった関数**(活性化関数)に通します。代表的なのはこの 2 つです。

- **シグモイド関数** $\sigma(z) = \dfrac{1}{1 + e^{-z}}$ … 出力を 0〜1 に押し込める
- **ReLU** $\mathrm{relu}(z) = \max(0, z)$ … 負なら 0、正ならそのまま。現在の主流

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def relu(z):
    return np.maximum(0, z)


print("sigmoid(z) =", sigmoid(z))
print("relu(z)    =", relu(z))

In [ ]:
t = np.linspace(-6, 6, 200)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(t, sigmoid(t))
axes[0].set_title("シグモイド関数")
axes[1].plot(t, relu(t), color="darkorange")
axes[1].set_title("ReLU")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.set_xlabel("z")
plt.tight_layout()
plt.show()

## 3. 層と順伝播 — 行列のかけ算で一気に計算

ニューロンを縦に並べたものが**層**です。今回作るのは、次の 2 層ネットワークです。

```
入力 X → [隠れ層: 重み W1, バイアス b1, ReLU] → [出力層: 重み W2, バイアス b2, softmax] → 予測
```

たくさんのニューロン・たくさんのデータの計算は、行列のかけ算 1 回にまとめられます。
データを行列 $X$(行がデータ、列が特徴量)にすると、隠れ層全体の計算は `X @ W1 + b1` と書けるのです。

出力層には **softmax 関数** を使います。これは出力の並びを「合計が 1 になる確率」に変換する関数で、
「クラス 0 の確率 70%、クラス 1 の確率 30%」のような答え方をさせるために使います。

In [ ]:
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)  # 大きな値でオーバーフローしないための工夫
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


print(softmax(np.array([[2.0, 1.0, 0.1]])))  # 合計すると 1 になる

重みの初期値は小さな乱数にします(全部 0 だと全ニューロンが同じ動きをしてしまいます)。
ReLU と相性のよい **He の初期化**(標準偏差 $\sqrt{2/入力数}$ の乱数)を使います。

`forward()` が**順伝播**、つまり入力から予測を計算する関数です。
途中の計算結果も返しているのは、後で勾配を求めるときに再利用するためです。

In [ ]:
def init_params(n_input, n_hidden, n_output, seed=0):
    rng = np.random.default_rng(seed)
    return {
        "W1": rng.normal(0, np.sqrt(2 / n_input), size=(n_input, n_hidden)),
        "b1": np.zeros(n_hidden),
        "W2": rng.normal(0, np.sqrt(2 / n_hidden), size=(n_hidden, n_output)),
        "b2": np.zeros(n_output),
    }


def forward(params, X):
    Z1 = X @ params["W1"] + params["b1"]  # 隠れ層の重み付き和
    A1 = relu(Z1)                         # 隠れ層の出力
    Z2 = A1 @ params["W2"] + params["b2"] # 出力層の重み付き和
    Y_hat = softmax(Z2)                   # 予測(各クラスの確率)
    return {"X": X, "Z1": Z1, "A1": A1, "Y_hat": Y_hat}

In [ ]:
params = init_params(n_input=2, n_hidden=3, n_output=2)

X_toy = np.array([[1.0, 0.5],
                  [0.2, -0.3]])
out = forward(params, X_toy)
print("予測(各行の合計が 1 になる確率):")
print(out["Y_hat"])

まだ学習していないので、予測は「五分五分」に近いデタラメです。ここからが本番です。

## 4. 損失関数 — どれくらい間違えたかを 1 つの数字にする

学習には「今の予測がどれくらい悪いか」を測るものさしが要ります。分類では**交差エントロピー**を使います。

$$L = -\frac{1}{N} \sum \log(\text{正解クラスに与えた確率})$$

正解クラスに確率 1.0 を与えれば損失 0、確率が低いほど損失は大きくなります。
正解ラベルは **one-hot 表現**(正解の位置だけ 1、他は 0 のベクトル)にしておくと計算が書きやすくなります。

In [ ]:
def one_hot(y, n_classes):
    Y = np.zeros((len(y), n_classes))
    Y[np.arange(len(y)), y] = 1
    return Y


def cross_entropy(Y_hat, Y):
    return float(-np.sum(Y * np.log(Y_hat + 1e-12)) / len(Y))


Y_toy = one_hot(np.array([0, 1]), 2)
print("one-hot 表現:")
print(Y_toy)
print("損失 =", round(cross_entropy(out["Y_hat"], Y_toy), 4))

## 5. 勾配降下法 — 損失が減る方向に少しずつ進む

学習の作戦はこうです:

> 各パラメータについて「少し増やしたら損失は増える? 減る?」(= **勾配**)を調べ、
> 損失が**減る**方向へ少しだけ動かす。これをひたすら繰り返す。

これが**勾配降下法**です。1 歩の大きさを決める係数を**学習率**と呼びます。
まずは単純な関数 $f(w) = (w - 3)^2$ で試してみましょう。勾配は $f'(w) = 2(w-3)$ です。

In [ ]:
w = -1.0          # 適当な初期値
lr = 0.1          # 学習率
history = [w]
for _ in range(30):
    grad = 2 * (w - 3)      # 勾配
    w = w - lr * grad       # 勾配の逆方向へ 1 歩
    history.append(w)

plt.figure(figsize=(7, 3))
plt.plot(history, marker="o", markersize=4)
plt.axhline(3, color="crimson", linestyle="--", label="最小値 w = 3")
plt.title("勾配降下法で最小値へ近づいていく")
plt.xlabel("ステップ")
plt.ylabel("w の値")
plt.legend()
plt.show()

## 6. 逆伝播 — 全パラメータの勾配を一度に求める

ネットワークには重みが何千個もあるので、1 個ずつ「少し動かして試す」のでは遅すぎます。
そこで微分の**連鎖律**を使い、出力側から入力側へ順に勾配を伝えていきます。これが**逆伝播**です。

うれしいことに、softmax + 交差エントロピーの組み合わせでは、出力層の勾配が驚くほど簡単になります。

$$\frac{\partial L}{\partial Z_2} = \hat{Y} - Y \quad (\text{予測} - \text{正解})$$

あとは連鎖律で 1 層ずつさかのぼるだけです(ReLU の微分は「正なら 1、負なら 0」)。

In [ ]:
def backward(params, cache, Y):
    N = len(Y)
    dZ2 = (cache["Y_hat"] - Y) / N          # 出力層: 予測 - 正解
    dW2 = cache["A1"].T @ dZ2
    db2 = dZ2.sum(axis=0)
    dA1 = dZ2 @ params["W2"].T              # 勾配を隠れ層へ伝える
    dZ1 = dA1 * (cache["Z1"] > 0)           # ReLU の微分
    dW1 = cache["X"].T @ dZ1
    db1 = dZ1.sum(axis=0)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}


def update(params, grads, lr):
    for key in params:
        params[key] -= lr * grads[key]

### 逆伝播は本当に正しい? — 数値微分でチェック

逆伝播の実装ミスは気づきにくいので、「重みを実際に少し動かして損失の変化を測る」**数値微分**と比べて確認します。
2 つの値がほぼ一致すれば、逆伝播は正しく書けています。

In [ ]:
def numerical_grad_W2(params, X, Y, i, j, eps=1e-5):
    params["W2"][i, j] += eps
    loss_plus = cross_entropy(forward(params, X)["Y_hat"], Y)
    params["W2"][i, j] -= 2 * eps
    loss_minus = cross_entropy(forward(params, X)["Y_hat"], Y)
    params["W2"][i, j] += eps  # 元に戻す
    return (loss_plus - loss_minus) / (2 * eps)


cache = forward(params, X_toy)
grads = backward(params, cache, Y_toy)
print("逆伝播の勾配  :", round(grads["W2"][0, 0], 8))
print("数値微分の勾配:", round(numerical_grad_W2(params, X_toy, Y_toy, 0, 0), 8))

## 7. 小さな問題で学習してみる — XOR

道具がそろったので学習させてみます。題材は **XOR**(2 つの入力が「違うときだけ 1」)。
XOR は 1 本の直線では分けられないため、隠れ層のないネットワークでは絶対に解けない、有名なテスト問題です。

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])
Y_xor = one_hot(y_xor, 2)

params = init_params(n_input=2, n_hidden=8, n_output=2, seed=1)
losses = []
for epoch in range(2000):
    cache = forward(params, X_xor)
    losses.append(cross_entropy(cache["Y_hat"], Y_xor))
    update(params, backward(params, cache, Y_xor), lr=0.5)

pred = forward(params, X_xor)["Y_hat"].argmax(axis=1)
print("予測:", pred)
print("正解:", y_xor)

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.title("XOR の学習曲線 — 損失が下がっていく")
plt.xlabel("エポック(繰り返し回数)")
plt.ylabel("損失")
plt.show()

ネットワークが入力平面をどう分けたか(**決定境界**)も見てみましょう。
平面を細かい格子で埋めて、各点の予測確率を色で塗ります。

In [ ]:
gx, gy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = np.c_[gx.ravel(), gy.ravel()]
proba = forward(params, grid)["Y_hat"][:, 1].reshape(gx.shape)

plt.figure(figsize=(5.5, 4))
plt.contourf(gx, gy, proba, levels=20, cmap="RdBu_r", alpha=0.7)
plt.colorbar(label="クラス 1 の確率")
plt.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap="RdBu_r",
            edgecolors="black", s=100, zorder=3)
plt.title("XOR の決定境界")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

直線 1 本では作れない、曲がった境界を学習できていることがわかります。

## 8. 手書き数字の分類に挑戦

いよいよ本番です。scikit-learn に同梱されている手書き数字データセット(**digits**)を使います。
8×8 ピクセル(= 64 個の数値)の画像が 1797 枚あり、それぞれに 0〜9 のラベルが付いています。
同梱データなのでダウンロードは不要です。

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
print("画像:", digits.images.shape, " ラベル:", digits.target.shape)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(8, 2.6))
for ax, img, label in zip(axes.ravel(), digits.images, digits.target):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(label, fontsize=9)
    ax.axis("off")
plt.suptitle("手書き数字の例(8×8 ピクセル)")
plt.tight_layout()
plt.show()

前処理をします。

- ピクセル値(0〜16)を 16 で割って 0〜1 に**正規化**
- データを**訓練用**(学習に使う)と**テスト用**(実力測定に使う)に分割
- 訓練用ラベルを one-hot 表現に変換

「テスト用を学習に使わない」のは、丸暗記ではなく本当に一般化できたかを測るためです。

In [ ]:
X = digits.data / 16.0
y = digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)
Y_train = one_hot(y_train, 10)
print("訓練データ:", X_train.shape, " テストデータ:", X_test.shape)

学習ループは XOR とまったく同じです。**さっき自分で書いた部品をそのまま使います**。
ネットワークは「入力 64 → 隠れ層 32 → 出力 10」。パラメータ数は約 2400 個です。

In [ ]:
def accuracy(params, X, y):
    pred = forward(params, X)["Y_hat"].argmax(axis=1)
    return float((pred == y).mean())


params = init_params(n_input=64, n_hidden=32, n_output=10, seed=0)
train_losses = []
test_accs = []

for epoch in range(600):
    cache = forward(params, X_train)
    train_losses.append(cross_entropy(cache["Y_hat"], Y_train))
    update(params, backward(params, cache, Y_train), lr=0.5)
    if (epoch + 1) % 20 == 0:
        test_accs.append((epoch + 1, accuracy(params, X_test, y_test)))

print(f"訓練データの正解率  : {accuracy(params, X_train, y_train):.1%}")
print(f"テストデータの正解率: {accuracy(params, X_test, y_test):.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(train_losses)
axes[0].set_title("損失の推移")
axes[0].set_xlabel("エポック")
axes[0].set_ylabel("損失")

epochs_, accs_ = zip(*test_accs)
axes[1].plot(epochs_, accs_, marker="o", markersize=4, color="seagreen")
axes[1].set_title("テスト正解率の推移")
axes[1].set_xlabel("エポック")
axes[1].set_ylabel("正解率")
plt.tight_layout()
plt.show()

NumPy だけで書いたネットワークが、95% 以上の精度で手書き数字を読めるようになりました。
実際の予測結果を見てみましょう(赤字は間違えたもの)。

In [ ]:
pred_test = forward(params, X_test)["Y_hat"].argmax(axis=1)

fig, axes = plt.subplots(2, 8, figsize=(8, 3))
for ax, img, p, t in zip(axes.ravel(), X_test, pred_test, y_test):
    ax.imshow(img.reshape(8, 8), cmap="gray_r")
    ax.set_title(f"予測 {p}", fontsize=9,
                 color="black" if p == t else "crimson")
    ax.axis("off")
plt.suptitle("テストデータの予測結果")
plt.tight_layout()
plt.show()

In [ ]:
wrong = np.where(pred_test != y_test)[0]
print(f"間違えた枚数: {len(wrong)} / {len(y_test)}")

n_show = min(8, len(wrong))
fig, axes = plt.subplots(1, n_show, figsize=(n_show, 1.9))
for ax, i in zip(np.atleast_1d(axes).ravel(), wrong[:n_show]):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"予測{pred_test[i]} 正解{y_test[i]}",
                 fontsize=8, color="crimson")
    ax.axis("off")
plt.suptitle("間違えた例 — 人間でも迷いそうな字が多い")
plt.tight_layout()
plt.show()

## 練習問題

ここまでの部品を使って、自分で実験してみましょう。

**練習 1**: 隠れ層のニューロン数を 32 から 64 に増やすと、テスト正解率はどう変わるでしょうか。

**練習 2**: 学習率 `lr` を 0.05 や 2.0 に変えると、学習曲線(損失の推移)はどうなるでしょうか。
小さすぎるとき・大きすぎるときの違いを観察してみてください。

In [ ]:
# ここで自由に実験してみましょう


### 解答例 1

In [ ]:
params64 = init_params(n_input=64, n_hidden=64, n_output=10, seed=0)
for epoch in range(600):
    cache = forward(params64, X_train)
    update(params64, backward(params64, cache, Y_train), lr=0.5)

print(f"隠れ層 64 個のテスト正解率: {accuracy(params64, X_test, y_test):.1%}")

## まとめと次のステップ

このノートブックで作ったもの:

- **順伝播** … 行列のかけ算 + 活性化関数(ReLU、softmax)で予測する
- **損失関数** … 交差エントロピーで「間違い具合」を 1 つの数字にする
- **逆伝播** … 連鎖律で全パラメータの勾配を一度に求める(数値微分で検算もした)
- **勾配降下法** … 勾配の逆方向へ少しずつ進む学習ループ

PyTorch や TensorFlow といった深層学習ライブラリがやっていることも、本質はこれと同じです
(自動微分・GPU 対応・便利な層の部品が加わっているだけです)。

### 次に学ぶなら

- **micrograd** … Andrej Karpathy 作の約 150 行の自動微分エンジン。逆伝播の理解が深まります。作者の解説動画(YouTube: The spelled-out intro to neural networks)も名作です
- **書籍『ゼロから作る Deep Learning』**(斎藤康毅)… このノートブックの内容をさらに深く、CNN まで扱います
- **PyTorch** … 基礎を理解した後の実用ライブラリ。`autograd` の仕組みが今日書いたコードの延長線上にあります(ブラウザでは動かないため、ローカル環境か Google Colab で)